# Guesser prepared-data audit

This notebook audits **labels and split manifests for production, medium, or small** without a trained model. It reads the selected profile's contract from the data package and checks the label registry, per-class split counts, manifest row shape, and drawing-ID / trajectory-hash disjointness. The uniqueness check is disk-backed so it does not load all IDs into notebook memory.

This notebook verifies manifests; it does not score a model. To verify indexed payload bytes, run the optional integrity-audit cell below (the export step in `01_dataset_pipeline.ipynb` already performs this check). To inspect image treatment, enable `INCLUDE_PREVIEW` in notebook 01 and inspect `<profile>/preview/index.html`. To reproduce the available A/B training-validation report, continue to `03_training.ipynb`; notebook 04 only checks for artifacts and cannot produce ten-stage scores. Keep held-out test labels out of checkpoint selection and threshold tuning.

Run from the repository root with `uv run --project models/guesser/evaluation jupyter lab --ServerApp.root_dir=.`. Do not add the notebooks folder as a positional argument: Jupyter would use that folder as the Contents API root. Open `models/guesser/evaluation/notebooks` in the file browser. Prepare the separate data environment first, when no data job is active. Profile discovery uses `uv run --frozen --no-sync` with Python 3.13 and never starts downloads or syncs dependencies; it first verifies the data package is installed in that environment and otherwise stops with the `uv sync` command to run. Re-run the audit cell after the data pipeline finishes. 

In [ ]:
# Reload our generated module after its writer cell runs, without restarting the kernel.
%load_ext autoreload
%autoreload 2

## Options and dataset location

| Option | Default | Effect |
|---|---|---|
| `PROFILE` | `None` | Use the data CLI default (currently production), or select `production`, `medium`, or `small`. Smoke is not trainable and is rejected. Expected counts are read from `settings.py`, not copied here. |
| `DATA_ROOT` | `None` | Use the selected profile's directory. An explicit path overrides `MTMAI_GUESSER_DATA_ROOT`, which overrides the default. Both overrides name the directory **containing `prepared/`, `labels.json`, and `dataset-summary.json`**, not its parent as in notebook 01. Relative overrides use the kernel working directory. |

The audit needs all three split manifests and rejects a summary whose profile differs from the selection. An extracted trainer ZIP has no test manifest and cannot pass this full split-isolation audit by itself. Keep the held-out package sealed until the model and evaluation protocol are frozen; do not extract it just to run this notebook early. Medium/small are prefix samples, not controlled subsets of production.

In [ ]:
# Resolve the helper path from this checkout so the notebook works from different cwd values.
from guesser_evaluation.notebook_tools import module_source_path

workflow_source_path = str(module_source_path("dataset_audit_workflow.py"))

# None delegates to the source defaults; explicit paths name a profile directory.
PROFILE = None
DATA_ROOT = None
print(f"Workflow source: {workflow_source_path}")

In [ ]:
%%writefile {workflow_source_path}
"""Notebook helpers for locating, auditing, and plotting Guesser data."""

from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from typing import Any

from .data_environment import require_data_environment


def data_profile(profile: str | None = None) -> dict[str, Any]:
    """Read the selected profile contract in the data project's own environment."""
    from .notebook_tools import module_source_path

    source_path = module_source_path("dataset_audit_workflow.py")
    data_project = source_path.parents[2].parent / "data"
    repository_root = data_project.parents[2]
    # Report an unsynced data environment directly instead of uv's spawn error.
    require_data_environment(data_project, cwd=repository_root)
    # Query settings without syncing or importing Python 3.13 code into this kernel.
    query = """
import json, sys
from guesser_data.cli import parser
from guesser_data.settings import PACKAGE_ROOT, Settings
defaults = parser().parse_args(["download"])
settings = Settings(profile=sys.argv[1] or defaults.profile)
if not settings.is_trainable:
    raise ValueError("Audit requires a trainable profile, not smoke")
print(json.dumps({
    "profile": settings.profile, "data_root": str(settings.base),
    "counts": {split: settings.spec[split] for split in ("train", "validation", "test")},
    "class_count": len((PACKAGE_ROOT / "src/resources/categories.txt").read_text().splitlines()),
    "sampling_method": settings.download_method,
}))
"""
    result = subprocess.run(
        [
            "uv",
            "run",
            "--project",
            str(data_project),
            "--frozen",
            "--no-sync",
            "python",
            "-c",
            query,
            "" if profile is None else profile,
        ],
        cwd=repository_root,
        check=True,
        capture_output=True,
        text=True,
    )
    return json.loads(result.stdout)


def profile_data_root(profile: str | None = None) -> Path:
    """Resolve a profile directory, preserving the legacy environment override."""
    contract = data_profile(profile)
    return Path(os.environ.get("MTMAI_GUESSER_DATA_ROOT", contract["data_root"])).expanduser()


def production_data_root() -> Path:
    """Keep the original production-only path helper compatible."""
    return profile_data_root("production")


def audit_dataset(
    data_root: str | Path | None = None, *, profile: str | None = None
) -> dict[str, Any] | None:
    """Audit a selected training profile, returning None while preparation is pending."""
    from .dataset_audit import audit_prepared_dataset

    contract = data_profile(profile)
    selected = contract["profile"] if profile is None else profile
    # Explicit paths and the legacy environment variable name a profile directory.
    configured = (
        data_root
        if data_root is not None
        else os.environ.get("MTMAI_GUESSER_DATA_ROOT", contract["data_root"])
    )
    root = Path(configured).expanduser()
    prepared_path = root / "prepared"
    # A missing folder means preparation is pending, not that the audit passed.
    if not prepared_path.is_dir():
        return None
    summary = json.loads((root / "dataset-summary.json").read_text(encoding="utf-8"))
    if summary.get("profile") != selected:
        raise ValueError(f"Dataset profile does not match selected profile {selected!r}")
    return audit_prepared_dataset(
        prepared_path,
        # Counts and registry size come from the data source, not duplicated constants.
        expected_class_count=contract["class_count"],
        expected_per_class=contract["counts"],
    )


def audit_production_dataset(data_root: str | Path | None = None) -> dict[str, Any] | None:
    """Preserve the existing production-only public entry point."""
    return audit_dataset(data_root, profile="production")


def plot_class_balance(report: dict[str, Any]):
    """Return per-split totals and a class-count figure from a passed audit."""
    import matplotlib.pyplot as plt
    import pandas as pd

    counts = pd.DataFrame(report["counts_by_class"])
    # Sum the same audited rows shown in the plot for a quick total check.
    totals = counts[["train", "validation", "test"]].sum()
    # Stack partitions so class balance can be compared without hiding any split.
    figure, axis = plt.subplots(figsize=(18, 5))
    counts.set_index("source_name")[["train", "validation", "test"]].plot(
        kind="bar", stacked=True, width=0.9, ax=axis
    )
    axis.set_title("Original drawings by Quick, Draw! class and split")
    axis.set_xlabel("Class")
    axis.set_ylabel("Original drawings")
    figure.tight_layout()
    return totals, figure


## Audit labels, counts, and split isolation

The selected profile's expected counts, label count, and sampling method are displayed below. The function reads Parquet manifests in batches and uses a temporary SQLite index to detect duplicate identities and trajectories across all splits. Missing preparation returns a pending state, not a passing audit. This reads test manifests when they are present in the selected dataset, but does not open ZIPs or read test drawing payloads.

In [ ]:
# Refresh discovery after %%writefile creates the reusable workflow module.
import importlib

importlib.invalidate_caches()
from guesser_evaluation.dataset_audit_workflow import audit_dataset, data_profile, profile_data_root

print("Selected profile contract:", data_profile(PROFILE))
print("Dataset directory:", DATA_ROOT if DATA_ROOT is not None else profile_data_root(PROFILE))
audit_report = audit_dataset(DATA_ROOT, profile=PROFILE)
if audit_report is None:
    # Missing data means acquisition/preparation is pending, not that validation passed.
    audit_report = None
    print("Prepared data is not available yet. Re-run this cell after download and prepare finish.")
audit_report

## Inspect class balance

The chart visualizes counts from the manifests after the audit passes. It is a dataset-structure check, not model performance. A failed audit raises an error above instead of silently omitting rows.

In [ ]:
from IPython.display import display

if audit_report is not None:
    # Only visualize class balance after every manifest check has passed.
    from guesser_evaluation.dataset_audit_workflow import plot_class_balance

    split_totals, class_balance_figure = plot_class_balance(audit_report)
    display(split_totals.to_frame(name="originals"))
    display(class_balance_figure)
else:
    print("No prepared manifests to summarize yet.")

## Optional indexed-payload integrity audit

The data package audit checks each indexed JSONL byte range, record/manifest agreement, split counts, file coverage, and the stored summary. It reads payload records from all three splits, including test, strictly for integrity; it does not load a model or compute recognition metrics. The full pass can take time for production. A temporary SQLite index is removed when the command finishes. This check is disabled by default.

This does not visually review rendered preprocessing. Use the data pipeline's optional preview/contact sheets for that separate check.

> **Expected warning from the audit subprocess below:** the message that `VIRTUAL_ENV=models/guesser/evaluation/.venv` does not match `models/guesser/data/.venv` and will be ignored is intentional. The Python 3.12 notebook kernel launches the audit in the separate Python 3.13 data project with `--project`; do not use `--active`. This exact mismatch alone is expected; other errors or nonzero exits still indicate a problem.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

from guesser_evaluation.notebook_tools import module_source_path

# Keep the full record scan opt-in; the manifest audit above is the default.
RUN_PAYLOAD_AUDIT = False
if not RUN_PAYLOAD_AUDIT:
    print("Payload audit disabled. Set RUN_PAYLOAD_AUDIT = True to scan indexed records.")
else:
    if audit_report is None:
        raise RuntimeError("Run a passing manifest audit before the payload audit.")

    # Use the exact profile directory audited above; the data CLI accepts its parent root.
    profile_directory = Path(audit_report["prepared_path"]).resolve().parent
    summary = json.loads((profile_directory / "dataset-summary.json").read_text(encoding="utf-8"))
    selected_profile = summary["profile"]
    if profile_directory.name != selected_profile:
        raise ValueError(
            f"Data CLI expects a {selected_profile!r} profile directory; got {profile_directory}."
        )

    # Remove host-injected Python paths before entering the separate data environment.
    data_environment = dict(os.environ)
    data_environment.pop("PYTHONPATH", None)
    data_environment.pop("PYTHONHOME", None)
    data_environment["PYTHONUNBUFFERED"] = "1"
    workflow_path = module_source_path("dataset_audit_workflow.py")
    data_project = workflow_path.parents[2].parent / "data"
    repository_root = data_project.parents[2]

    # Probe readiness first; all subprocesses use --no-sync to avoid hidden environment changes.
    probe = subprocess.run(
        [
            "uv",
            "run",
            "--project",
            str(data_project),
            "--frozen",
            "--no-sync",
            "python",
            "-c",
            "import guesser_data",
        ],
        cwd=repository_root,
        env=data_environment,
        capture_output=True,
        check=False,
        text=True,
    )
    if probe.returncode != 0:
        detail = probe.stderr.strip() or probe.stdout.strip()
        raise RuntimeError(
            "Data environment is not ready. Run `uv sync --project models/guesser/data --frozen` "
            f"from the repository root, then retry.\n{detail}"
        )

    # Progress remains visible on stderr; capture the final JSON summary on stdout.
    audit_command = [
        "uv",
        "run",
        "--project",
        str(data_project),
        "--frozen",
        "--no-sync",
        "guesser-data",
        "audit",
        "--profile",
        selected_profile,
        "--data-root",
        str(profile_directory.parent),
        "--json",
    ]
    print("[stage] auditing indexed payloads across all three splits", flush=True)
    result = subprocess.run(
        audit_command,
        cwd=repository_root,
        env=data_environment,
        stdout=subprocess.PIPE,
        check=False,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Data payload audit failed:\n{result.stdout.strip()}")
    audit_result = json.loads(result.stdout)["result"]
    print("Payload audit:", audit_result["status"])
    print("Profile:", selected_profile)
    print("Summary:", profile_directory / "dataset-summary.json")

## Next: reproduce the current A/B training comparison

1. If prepared data is missing, return to `01_dataset_pipeline.ipynb` and finish download/prepare/export. If this manifest audit fails, fix the reported data problem there, rerun prepare/export, and then rerun this audit; do not start training from an incomplete or failed audit.
2. If it passes, open `03_training.ipynb`. Set `DATA_PROFILE` to the resolved profile shown in this notebook's output (`PROFILE=None` resolves to the data CLI default, currently `production`) and use that profile's `prepared/` data. To reproduce the small-profile model run, use `small` in notebooks 01, 02, and 03.
3. In notebook 03, choose full mode and both A/B runs, then enable `RUN_TRAINING`. After both complete, enable `RUN_COMPARISON` to write the validation report and plots under the selected checkpoint root's `ab_comparison/` directory. Keep the profile, seed, preprocessing, training settings, and checkpoint root the same for A and B.
4. The optional payload audit above is only needed to recheck an existing dataset; export already checks payload integrity. Use notebook 01's preview when you need to inspect rendered/preprocessed examples. Notebook 04 checks whether A/B artifacts exist, but the repository has no ten-stage prediction/scoring runner yet, so those recognition scores cannot currently be reproduced.

The cell outputs above include the results of the selected manifest and payload checks. The A/B comparison is the current runnable model result; it uses training/validation logs and is not a held-out recognition evaluation. Keep test labels out of model selection and threshold tuning.